# Imports

## Standard Library

In [40]:
import os          # file paths, directory management
import sys         # system-level operations (rarely used, but useful for debugging)
import time        # timing execution (performance measurement)
import gc          # garbage collection (manual memory cleanup)

## Data Handling & Numerical Computation

In [41]:
import numpy as np         # numerical operations, arrays, embeddings
import pandas as pd        # dataset loading and manipulation (DataFrames)

## System Monitoring & External Communication

In [42]:
import psutil              # monitor RAM/CPU usage (GreenAI metrics)
import requests            # send notifications (by NTFY)

## Image Processing & Computer Vision

In [43]:
from PIL import Image      # image loading
import cv2                 # OpenCV (advanced image processing if needed)

## Progress & Visualization

In [44]:
from tqdm.notebook import tqdm   # progress bars

import matplotlib.pyplot as plt  # plotting results
import seaborn as sns            # statistical visualizations

## Statistics / Evaluation

In [45]:
from scipy.stats import spearmanr   # rank correlation (similarity preservation)
import pickle 

## PyTorch (Core Deep Learning Framework)

In [46]:
import torch
import torch.nn as nn              # neural network layers

## TorchVision (Pretrained Vision Models & Transforms)

In [47]:
from torchvision import models, transforms

## Transformers (HuggingFace Models)

In [48]:
from transformers import (
    # Vision Transformers
    ViTModel, ViTImageProcessor,
    DeiTModel, DeiTImageProcessor,

    # Generic auto models (flexible loading)
    AutoImageProcessor, AutoModel,

    # CLIP (multimodal)
    CLIPProcessor, CLIPVisionModel, CLIPTextModel,

    # Text models
    BertTokenizer, BertModel,
    RobertaTokenizer, RobertaModel,
    GPT2Tokenizer, GPT2Model
)

# Configuration

## Dataset Selection

In [49]:
# Available datasets: Flickr8k, Flickr30k, ConceptualCaptions
CURRENT_DATASET = "Flickr8k" 
ALL_DATASETS = ["Flickr8k", "Flickr30k", "ConceptualCaptions"]

assert CURRENT_DATASET in ALL_DATASETS, f"{CURRENT_DATASET} is not a valid dataset"


## Directory Architecture

In [50]:

BASE_DIR = os.path.join(os.getcwd(), 'TFE_Data')

DATASETS_DIR = os.path.join(BASE_DIR, 'Datasets')
RAW_DIR = os.path.join(BASE_DIR, 'Features_RAW', CURRENT_DATASET)
RESULTS_DIR = os.path.join(BASE_DIR, 'Results', CURRENT_DATASET)

def get_raw_dir(dataset, modality):
    path = os.path.join(BASE_DIR, "Features", "raw", dataset, modality)
    os.makedirs(path, exist_ok=True)
    return path

def get_results_dir(dataset):
    path = os.path.join(BASE_DIR, "Results", dataset)
    os.makedirs(path, exist_ok=True)
    return path

## Saliency Maps Directory (per dataset)

In [51]:
def get_saliency_dir(dataset_name, model_name):
    path = os.path.join(BASE_DIR, "Results", dataset_name, model_name, "SaliencyMaps")
    os.makedirs(path, exist_ok=True)
    return path

## Device Configuration

In [52]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Environment initialized. Using device: {device}")

Environment initialized. Using device: cuda


## Notification Setup

In [53]:
def send_ntfy_notification(message, title="TFE Pipeline Update"):
    """Sends a push notification via ntfy.sh."""
    NTFY_TOPIC = "aysel_tfe_server_9988"
    try:
        requests.post(
            f"https://ntfy.sh/{'aysel_tfe_server_9988'}",
            data=message.encode(encoding='utf-8'),
            headers={"Title": title}
        )
    except Exception as e:
        print(f"Notification failed: {e}")



# Data Loading

In [54]:
print(f"Loading dataset: {CURRENT_DATASET}...")

df_path = os.path.join(DATASETS_DIR, f"df_{CURRENT_DATASET}.pkl")
if not os.path.exists(df_path):
    raise FileNotFoundError(f"Dataset file not found at {df_path}. Please run data_builder.ipynb first.")

df = pd.read_pickle(df_path)

# Extract inputs for models
IMAGE_PATHS = df['image_path'].tolist()

CAPTIONS_LIST = df['captions'].tolist()
assert len(IMAGE_PATHS) == len(CAPTIONS_LIST) # sanity check: each image should have a list of captions
print(f"Ready. Loaded {len(IMAGE_PATHS)} images.")
print(f"Each image has {len(CAPTIONS_LIST[0])} captions (example).")

texts_for_xai = [caps[0] for caps in CAPTIONS_LIST] # For XAI, we will use the first caption of each image to keep it simple.
print(f"Ready. Loaded {len(IMAGE_PATHS)} images and {len(texts_for_xai)} captions into memory.")

Loading dataset: Flickr8k...
Ready. Loaded 8091 images.
Each image has 5 captions (example).
Ready. Loaded 8091 images and 8091 captions into memory.


# Utility Functions

## GreenAI Metrics

In [55]:
def measure_memory():
    """Returns the current memory usage of the process in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

In [56]:
def get_size_in_mb(obj):
    """Returns the size of a numpy array in MB."""
    if isinstance(obj, np.ndarray):
        return obj.nbytes / (1024 * 1024)
    else:
        return sys.getsizeof(obj) / (1024 * 1024)

In [57]:
def save_metadata(data, dataset, modality, model_name):
    model_dir = os.path.join(BASE_DIR, "Results", dataset, modality, model_name)
    os.makedirs(model_dir, exist_ok=True)
    meta_path = os.path.join(model_dir, "metadata.pkl")
    with open(meta_path, "wb") as f:
        pickle.dump({"samples": data, "indices": list(range(len(data)))}, f)

In [58]:
def execute_and_save(dataset, modality, model_name, extract_func, data, device):
    """
    Runs feature extraction, saves embeddings, metadata, and returns GreenAI metrics.
    """
    start_time = time.time()
    mem_before = measure_memory()
    # Optional energy monitoring could be inserted here
    try:
        features = extract_func(data, device, dataset)
    except Exception as e:
        print(f"[ERROR] {model_name} extraction failed: {e}")
        return None

    exec_time = time.time() - start_time
    mem_used = max(0, measure_memory() - mem_before)
    latency = exec_time / len(data)
    throughput = len(data) / exec_time

    # --- Save embeddings ---
    model_dir = os.path.join(BASE_DIR, "Results", dataset, modality, model_name)
    os.makedirs(model_dir, exist_ok=True)
    emb_path = os.path.join(model_dir, "embeddings.npy")
    np.save(emb_path, features)

    # --- Save metadata ---
    save_metadata(data, dataset, modality, model_name)

    print(f"[SAVED] {model_name} | shape={features.shape} | dataset={dataset}")

    return {
        "Dataset": dataset,
        "Modality": modality,
        "Model": model_name,
        "Dim": features.shape[1],
        "Time_s": exec_time,
        "Latency_s": latency,
        "Throughput_samples_per_s": throughput,
        "Memory_MB": mem_used,
        "Energy_J": None  # optional
    }

## XAI Saliency Maps

### Vision XAI

In [59]:
def save_vision_saliency_and_overlay(model_name, activations, img_path, dataset_name):
    """
    Saves saliency maps and overlay images for vision models.
    Supports norm-based weighting or Grad-CAM if gradients exist.
    """
    act = activations.detach().cpu()
    
    # Compute saliency
    if act.ndim == 4:  # CNN: [B,C,H,W]
        weights = torch.norm(act, dim=1, keepdim=True)
        heatmap = torch.sum(act * weights, dim=1).squeeze().numpy()
    elif act.ndim == 3:  # Transformer: [B,T,D]
        tokens = act[:, 1:, :]
        if tokens.shape[1] in [197, 198]:
            tokens = tokens[:, :196, :]
        weights = torch.norm(tokens, dim=-1, keepdim=True)
        weighted_sum = torch.sum(tokens * weights, dim=-1)
        side = int(np.sqrt(tokens.shape[1]))
        heatmap = weighted_sum.view(side, side).numpy()
    else:
        return

    # Normalize & resize
    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) > 0:
        heatmap /= np.max(heatmap)
    heatmap_res = cv2.resize(heatmap, (224, 224))

    # Save raw saliency
    saliency_dir = get_saliency_dir(dataset_name, model_name)
    os.makedirs(saliency_dir, exist_ok=True)
    image_filename = os.path.splitext(os.path.basename(img_path))[0]
    np.save(os.path.join(saliency_dir, f"Saliency_{image_filename}.npy"), heatmap_res)

    # Save overlay
    overlay_dir = os.path.join(BASE_DIR, "Results", dataset_name, model_name, "Overlay")
    os.makedirs(overlay_dir, exist_ok=True)

    # --- FIX: Ensure correct path from DataFrame ---
    if not os.path.exists(img_path):
        print(f"[WARNING] Could not read image for overlay (wrong path?): {img_path}")
        return  # skip this image safely

    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        print(f"[WARNING] Could not read image for overlay (cv2 failed): {img_path}")
        return

    img_bgr = cv2.resize(img_bgr, (224, 224))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_res), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(img_bgr, 0.6, heatmap_color, 0.4, 0)
    cv2.imwrite(os.path.join(overlay_dir, f"Overlay_{image_filename}.jpg"), overlay)

### Text XAI

In [60]:
# Convert each caption to separate sample and keep mapping to image
def prepare_text_samples(df):
    texts = []
    caption_image_map = []
    for idx, caps in enumerate(df['captions']):
        if isinstance(caps, list):
            for cap in caps:
                texts.append(cap)
                caption_image_map.append(df['image_path'][idx])
        else:
            texts.append(caps)
            caption_image_map.append(df['image_path'][idx])
    return texts, caption_image_map

In [61]:
def save_text_saliency(tokens, scores, dataset, model_name, idx, img_path=None):
    saliency_dir = get_saliency_dir(dataset, model_name)
    path = os.path.join(saliency_dir, f"SaliencyText_{idx}.npy")
    np.save(path, {
        "tokens": tokens,
        "scores": scores,
        "image_path": img_path
    })

# Unimodal Models

## CBIR: Vision Feature Extractions

### Image Dataset

In [62]:
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, transform=None):
        # Filter out missing images at init
        self.image_paths = [p for p in image_paths if os.path.exists(p)]
        self.transform = transform
        if len(self.image_paths) == 0:
            raise ValueError("No valid image paths found.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"[WARNING] Could not read image: {img_path} | {e}")
            image = Image.new('RGB', (224, 224), (0, 0, 0))  # fallback

        if self.transform:
            try:
                image = self.transform(image)
            except Exception as e:
                print(f"[WARNING] Transform failed for image: {img_path} | {e}")
                image = torch.zeros((3, 224, 224))  # fallback tensor

        return image

### Feature Extraction

In [63]:
def vision_extract_features(model, model_name, dataloader, device, target_layer,
                            paths, dataset_name, save_saliency=True, gradcam=False):
    """
    Extract vision features and optionally save saliency maps (CNN or Transformer).
    """
    model.eval()
    features_list = []
    activations = {}
    img_idx = 0

    # Forward hook to capture intermediate activations
    def hook_fn(module, input, output):
        if hasattr(output, "last_hidden_state"):
            activations['value'] = output.last_hidden_state
        elif isinstance(output, tuple):
            activations['value'] = output[0]
        else:
            activations['value'] = output

    hook = target_layer.register_forward_hook(hook_fn) if target_layer else None

    for batch in tqdm(dataloader, desc=f"{model_name}"):
        batch = batch.to(device)
        batch_acts = None

        output = model(batch)
        batch_acts = activations.get('value')

        # --- Feature vector extraction ---
        if hasattr(output, 'last_hidden_state'):
            feat = output.last_hidden_state[:, 0, :]
        elif isinstance(output, torch.Tensor):
            if output.ndim == 4:  # CNN
                feat = torch.nn.functional.adaptive_avg_pool2d(output, 1).view(output.size(0), -1)
            else:
                feat = output
        else:
            feat = output[0]

        features_list.append(feat.detach().cpu().numpy())

        # --- Compute gradients if Grad-CAM ---
        if gradcam and batch_acts is not None:
            batch_acts.requires_grad_(True)
            target_score = feat.sum()
            target_score.backward(retain_graph=True)

        # --- Save saliency maps ---
        if save_saliency and batch_acts is not None:
            for i in range(batch_acts.size(0)):
                img_path = paths[img_idx]

                # --- FIX: Check that path exists ---
                if not os.path.exists(img_path):
                    print(f"[WARNING] Skipping missing image: {img_path}")
                    img_idx += 1
                    continue

                save_vision_saliency_and_overlay(
                    model_name.lower(),
                    batch_acts[i:i+1],
                    img_path,
                    dataset_name
                )
                img_idx += 1
            activations['value'] = None

        model.zero_grad()
        if batch_acts is not None and batch_acts.grad is not None:
            batch_acts.grad.zero_()

    if hook:
        hook.remove()

    return np.vstack(features_list)

#### CNN Embeddings

In [64]:
def get_resnet50_embeddings(image_paths, device, dataset_name):
    try:
        model = models.resnet50(weights="DEFAULT").to(device)
        target_layer = model.layer4[-1]
        model.fc = nn.Identity()  # Remove classifier

        transform = models.ResNet50_Weights.DEFAULT.transforms()
        dataset = ImageDataset(image_paths, transform=transform)
        loader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)

        return vision_extract_features(model, "resnet50", loader, device,
                                              target_layer, image_paths, dataset_name, save_saliency=True, gradcam=True)
    except Exception as e:
        print(f"[ERROR] ResNet50 embedding failed: {e}")
        return None

In [65]:
def get_mobilenet_v3_embeddings(image_paths, device, dataset_name):
    try:
        model = models.mobilenet_v3_large(weights="DEFAULT").to(device)
        target_layer = model.features[-1]
        model.classifier = nn.Identity()  # Remove classifier

        transform = models.MobileNet_V3_Large_Weights.DEFAULT.transforms()
        dataset = ImageDataset(image_paths, transform=transform)
        loader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)

        return vision_extract_features(model, "mobilenet_v3", loader, device,
                                              target_layer, image_paths, dataset_name, save_saliency=True, gradcam=True)
    except Exception as e:
        print(f"[ERROR] MobileNetV3 embedding failed: {e}")
        return None

#### Transformer Embeddings

In [89]:
def get_vit_embeddings(image_paths, device, dataset_name):
    try:
        from transformers import ViTImageProcessor, ViTModel
        processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
        model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device)

        target_layer = model.encoder.layer[-1]

        def collate_fn(batch):
            processed_imgs = []
            for img in batch:
                # Ensure image is a PIL Image
                try:
                    if not isinstance(img, Image.Image):
                        raise ValueError("Invalid image type")
                    img = img.convert('RGB')  # force RGB
                    pixel_values = processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
                except Exception as e:
                    # Fallback black RGB image if something fails
                    print(f"[WARNING] Replacing corrupted image with fallback | {e}")
                    fallback = Image.new('RGB', (224, 224), (0, 0, 0))
                    pixel_values = processor(images=fallback, return_tensors="pt")['pixel_values'].squeeze(0)
                processed_imgs.append(pixel_values)
            return torch.stack(processed_imgs)

        dataset = ImageDataset(image_paths)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=32, num_workers=4, collate_fn=collate_fn
        )

        return vision_extract_features(
            model, "vit", loader, device,
            target_layer, image_paths, dataset_name,
            save_saliency=True, gradcam=True
        )

    except Exception as e:
        print(f"[ERROR] ViT embedding failed: {e}")
        return None

In [67]:
def get_deit_embeddings(image_paths, device, dataset_name):
    try:
        processor = DeiTImageProcessor.from_pretrained('facebook/deit-base-distilled-patch16-224')
        model = DeiTModel.from_pretrained('facebook/deit-base-distilled-patch16-224').to(device)
        target_layer = model.encoder.layer[-1].output

        def collate_fn(batch):
            return torch.stack([processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0) for img in batch])

        dataset = ImageDataset(image_paths)
        loader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4, collate_fn=collate_fn)

        return vision_extract_features(model, "deit", loader, device,
                                              target_layer, image_paths, dataset_name, save_saliency=True, gradcam=True)
    except Exception as e:
        print(f"[ERROR] DeiT embedding failed: {e}")
        return None

In [68]:

def get_clip_vision_embeddings(image_paths, device, dataset_name):
    try:
        processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
        model = CLIPVisionModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
        target_layer = model.vision_model.encoder.layers[-1]

        def collate_fn(batch):
            return torch.stack([processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0) for img in batch])

        dataset = ImageDataset(image_paths)
        loader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4, collate_fn=collate_fn)

        return vision_extract_features(model, "clip_vision", loader, device,
                                              target_layer, image_paths, dataset_name, save_saliency=True, gradcam=True)
    except Exception as e:
        print(f"[ERROR] CLIP Vision embedding failed: {e}")
        return None

## T2T: Text Feature Extractions


### Text Dataset

In [69]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {key: val.squeeze(0) for key, val in encoding.items()}

### Multiple Captions Encoder

In [70]:
def encode_multiple_captions(captions, model, tokenizer, device):
    """
    Encodes multiple captions for the same image and averages the embeddings.
    """
    embeddings = []
    for cap in captions:
        inputs = tokenizer(
            cap, return_tensors="pt", truncation=True, padding=True
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1)  # mean pooling
            embeddings.append(emb.detach().cpu().numpy())
    return np.mean(embeddings, axis=0)

### Text Feature Extraction

In [71]:
def text_extract_features(model, tokenizer, dataloader, device, dataset_name, model_name, 
                          feature_type='mean_pool', save_xai=True, caption_image_map=None):
    """
    Extract embeddings for all captions and optionally save token-level saliency maps.
    Supports linking saliency to both caption and original image.
    """
    model.eval()
    features_list = []
    xai_index = 0

    for batch_idx, batch in enumerate(tqdm(dataloader, desc=f"{model_name} Text Extraction")):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)

        # Feature extraction
        if feature_type == 'mean_pool':
            mask = batch['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
            sum_emb = torch.sum(outputs.last_hidden_state * mask, dim=1)
            sum_mask = torch.clamp(mask.sum(1), min=1e-9)
            batch_features = (sum_emb / sum_mask).detach().cpu().numpy()
        else:
            batch_features = outputs.last_hidden_state[:, 0, :].detach().cpu().numpy()
        features_list.append(batch_features)

        # Token-level XAI
        if save_xai:
            for i in range(outputs.last_hidden_state.size(0)):
                tokens = tokenizer.convert_ids_to_tokens(batch['input_ids'][i])
                scores = outputs.last_hidden_state[i].norm(dim=-1).detach().cpu().numpy()
                img_path = caption_image_map[xai_index] if caption_image_map is not None else None
                save_text_saliency(tokens, scores, dataset_name, model_name, xai_index, img_path=img_path)
                xai_index += 1

    return np.vstack(features_list)

### Specialized RNN for text

In [72]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        output, (hidden, cell) = self.rnn(embedded)
        return hidden[-1], output  # last hidden for embedding, full seq for XAI

# ----------------------------
# RNN Embedding + XAI
# ----------------------------
def get_rnn_embeddings(texts, device, dataset_name, caption_image_map=None):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = SimpleRNN(tokenizer.vocab_size).to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)

    features_list = []
    saliency_dir = os.path.join(BASE_DIR, f"TextSaliency_{dataset_name}")
    os.makedirs(saliency_dir, exist_ok=True)

    model.eval()
    for idx, batch in enumerate(tqdm(dataloader, desc="RNN XAI")):
        input_ids = batch['input_ids'].to(device)
        with torch.no_grad():
            last_hidden, full_seq = model(input_ids)
            features_list.append(last_hidden.detach().cpu().numpy())

        # Token-level saliency: magnitude of LSTM output
        for b in range(full_seq.size(0)):
            tokens = tokenizer.convert_ids_to_tokens(input_ids[b])
            scores = full_seq[b].norm(dim=-1).detach().cpu().numpy()
            saliency_path = os.path.join(saliency_dir, f"SaliencyText_rnn_{idx*32 + b}.npy")
            np.save(saliency_path, {'tokens': tokens, 'scores': scores})

    return np.vstack(features_list)

### Text Models

In [73]:
def get_bert_embeddings(texts, device, dataset_name, caption_image_map=None):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return text_extract_features(model, tokenizer, dataloader, device, dataset_name, "bert")


In [74]:

def get_roberta_embeddings(texts, device, dataset_name, caption_image_map=None):
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    model = RobertaModel.from_pretrained('roberta-base').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return text_extract_features(model, tokenizer, dataloader, device, dataset_name, "roberta")


In [75]:

def get_gpt2_embeddings(texts, device, dataset_name, caption_image_map=None):
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2Model.from_pretrained('gpt2').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return text_extract_features(model, tokenizer, dataloader, device, dataset_name, "gpt2")


In [76]:
def get_clip_text_embeddings(texts, device, dataset_name=None, caption_image_map=None, max_length=32):
    from transformers import CLIPTextModel, CLIPTokenizer
    import torch
    import numpy as np
    from tqdm import tqdm

    tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
    model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

    features = []
    model.eval()
    batch_size = 32
    global_index = 0  # Tracks position across all batches

    for start_idx in tqdm(range(0, len(texts), batch_size), desc="Extracting CLIP Text"):
        batch_texts = texts[start_idx:start_idx+batch_size]

        encoding = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=int(max_length)
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoding)
            batch_features = outputs.last_hidden_state[:, 0, :].detach().cpu().numpy()  # CLS token
            features.append(batch_features)

            # --- Token-level saliency ---
            for b in range(outputs.last_hidden_state.size(0)):
                if caption_image_map is not None:
                    img_path = caption_image_map[global_index]
                else:
                    img_path = None

                tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'][b])
                scores = outputs.last_hidden_state[b].norm(dim=-1).detach().cpu().numpy()
                save_text_saliency(tokens, scores, dataset_name, "clip_text", global_index, img_path=img_path)
                global_index += 1

    return np.vstack(features)

# Execution

In [82]:
import os
import urllib.request
import zipfile
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# Base Directory
DATA_DIR = os.path.join(os.getcwd(), 'TFE_Data')

# --- NEW ARCHITECTURE ---
DATASETS_DIR = os.path.join(DATA_DIR, 'Datasets')
RAW_DIR = os.path.join(DATA_DIR, 'Features_RAW')
REDUCED_DIR = os.path.join(DATA_DIR, 'Features_Reduced')
RESULTS_DIR = os.path.join(DATA_DIR, 'Results')

# Create all necessary directories
for d in [DATA_DIR, DATASETS_DIR, RAW_DIR, REDUCED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Local Data Directory set to: {DATA_DIR}")
print("Subdirectories created: Datasets/, Features_RAW/, Features_Reduced/, Results/")

# CRITICAL: GitHub Security (.gitignore)
gitignore_path = os.path.join(os.getcwd(), '.gitignore')
with open(gitignore_path, 'a+') as f:
    f.seek(0)
    content = f.read()
    if 'TFE_Data/' not in content:
        f.write("\n# Ignore Dataset Folder\nTFE_Data/\n")
        f.write("__pycache__/\n*.npy\n*.zip\n")
print(".gitignore security is active.")

def download_image_task(item):
    """Function executed by threads to download a single image."""
    index, url, caption, img_dir = item
    img_name = f"gcc_50k_{index}.jpg"
    full_path = os.path.join(img_dir, img_name)
    
    if os.path.exists(full_path):
        return {"success": True, "image_name": img_name, "image_path": full_path, "caption": caption}
        
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    try:
        response = requests.get(url, headers=headers, timeout=4)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert('RGB')
        image.save(full_path, format="JPEG", quality=85)
        return {"success": True, "image_name": img_name, "image_path": full_path, "caption": caption}
    except:
        return {"success": False}

def build_conceptual_captions_50k(target_size=50000, max_workers=32):
    """Downloads exactly 50,000 valid images from the Conceptual Captions dataset."""
    print(f"\n--- BUILDING CONCEPTUAL CAPTIONS ({target_size} images) ---")
    dataset_dir = os.path.join(DATA_DIR, "ConceptualCaptions")
    img_dir = os.path.join(dataset_dir, "Images")
    os.makedirs(img_dir, exist_ok=True)
    
    try:
        from datasets import load_dataset
    except ImportError:
        import subprocess
        subprocess.check_call(["pip", "install", "datasets"])
        from datasets import load_dataset

    print("1. Loading HuggingFace metadata...")
    ds = load_dataset("conceptual_captions", split="train")
    
    print("2. Selecting a pool of 100,000 URLs to account for dead links...")
    ds_subset = ds.select(range(100000))
    tasks = [(i, url, cap, img_dir) for i, (url, cap) in enumerate(zip(ds_subset['image_url'], ds_subset['caption']))]
        
    data = []
    successful_downloads = 0
    
    print(f"3. Starting download with {max_workers} threads...")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(download_image_task, task): task for task in tasks}
        pbar = tqdm(total=target_size, desc="Valid images downloaded")
        
        for future in as_completed(futures):
            res = future.result()
            if res["success"]:
                data.append({
                    "image_name": res["image_name"],
                    "image_path": res["image_path"],
                    "caption": res["caption"]
                })
                successful_downloads += 1
                pbar.update(1)
                
            if successful_downloads >= target_size:
                print(f"\nTarget of {target_size} reached. Stopping current downloads...")
                for f in futures:
                    f.cancel()
                break
                
        pbar.close()

    print("\n4. Formatting and saving the DataFrame...")
    df_flat = pd.DataFrame(data)
    df = df_flat.groupby(['image_name', 'image_path'])['caption'].apply(list).reset_index()
    df.rename(columns={'caption': 'captions'}, inplace=True)
    
    save_path = os.path.join(DATASETS_DIR, 'df_ConceptualCaptions.pkl')
    df.to_pickle(save_path) 
    
    print(f"Process complete. The dataset of {len(df)} images is ready in {save_path}")
    return "ConceptualCaptions"

Local Data Directory set to: /home/aysel/tfe/TFE_Data
Subdirectories created: Datasets/, Features_RAW/, Features_Reduced/, Results/
.gitignore security is active.


In [ ]:
ALL_DATASETS = ["Flickr8k", "Flickr30k", "ConceptualCaptions"]
all_metrics = []

#dataset_name = build_conceptual_captions_50k(target_size=50000, max_workers=32)

# Load the DataFrame
df_path = os.path.join(DATASETS_DIR, 'df_ConceptualCaptions.pkl')
df = pd.read_pickle(df_path)

# IMAGE_PATHS and TEXTS ready for your pipeline
IMAGE_PATHS = df['image_path'].tolist()           # for vision models
CAPTIONS_LIST = df['captions'].tolist()          # list of lists for multiple captions

for ds_name in ALL_DATASETS:
    df_path = os.path.join(DATASETS_DIR, f"df_{ds_name}.pkl")
    if not os.path.exists(df_path):
        print(f"Skipping {ds_name}: Metadata not found.")
        continue

    df = pd.read_pickle(df_path)
    IMAGE_PATHS = df['image_path'].tolist()
    texts_for_xai, caption_image_map = prepare_text_samples(df)

    send_ntfy_notification(f"Starting Indexation for {ds_name}", "TFE Pipeline")

    # --- Vision Pipeline ---
    vision_pipeline = {
        "resnet50": get_resnet50_embeddings,
        "mobilenet_v3": get_mobilenet_v3_embeddings,
        "vit": get_vit_embeddings,
        "clip_vision": get_clip_vision_embeddings
    }

    all_metrics = []

    for name, func in vision_pipeline.items():
        metrics = execute_and_save(dataset_name, "vision", name, func, IMAGE_PATHS, device)
        if metrics: 
            all_metrics.append(metrics)

    # --- Text Pipeline ---
    text_pipeline = {
        "bert": get_bert_embeddings,
        "roberta": get_roberta_embeddings,
        "gpt2": get_gpt2_embeddings,
        "clip_text": get_clip_text_embeddings
    }

    for name, func in text_pipeline.items():
        # Flatten captions if model expects single caption per sample
        flattened_captions = [cap for caps in CAPTIONS_LIST for cap in caps]
        
        metrics = execute_and_save(dataset_name, "text", name, func, flattened_captions, device)
        if metrics:
            all_metrics.append(metrics)

    send_ntfy_notification(f"Completed {ds_name}", "TFE Pipeline Success")

# --- Save Global Metrics ---
os.makedirs(os.path.join(BASE_DIR, "Results"), exist_ok=True)
df_final = pd.DataFrame(all_metrics)
df_final.to_pickle(os.path.join(BASE_DIR, "Results", "global_unimodal_metrics.pkl"))

print("All datasets and models processed successfully.")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

vit:   0%|          | 0/1563 [00:00<?, ?it/s]

/tmp/ipykernel_3513692/2601490436.py:69: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:492.)
  if batch_acts is not None and batch_acts.grad is not None:
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


[WARNING] Replacing corrupted image with fallback | mean must have 1 elements if it is an iterable, got 3


The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


...

gpt2 Text Extraction:   0%|          | 0/1563 [00:00<?, ?it/s]

[SAVED] gpt2 | shape=(50000, 768) | dataset=ConceptualCaptions


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.

[SAVED] clip_text | shape=(50000, 512) | dataset=ConceptualCaptions
All datasets and models processed successfully.


# Need to clean
import matplotlib.pyplot as plt

def validate_saliency_maps(dataset_name, num_images=3):
    """
    Displays a comparison grid: Original Image vs. 5 Model Saliency Maps.
    """
    vision_models = ["resnet50", "mobilenet_v3", "vit", "deit", "clip_vision"]
    saliency_dir = os.path.join(BASE_DIR, f"SaliencyMaps_{dataset_name}")
    
    # Load dataset metadata to get original image paths
    df_path = os.path.join(DATASETS_DIR, f"df_{dataset_name}.pkl")
    df_temp = pd.read_pickle(df_path)
    
    for idx in range(num_images):
        img_path = df_temp.iloc[idx]['image_path']
        img_rgb = Image.open(img_path).convert('RGB')
        
        fig, axes = plt.subplots(1, 6, figsize=(20, 5))
        fig.suptitle(f"Dataset: {dataset_name} | Image Index: {idx}", fontsize=16)
        
        # 0. Original Image
        axes[0].imshow(img_rgb)
        axes[0].set_title("Original")
        axes[0].axis('off')
        
        # 1-5. Model Heatmaps
        for i, model in enumerate(vision_models):
            npy_file = os.path.join(saliency_dir, f"Saliency_{model}_{idx}.npy")
            
            if os.path.exists(npy_file):
                heatmap = np.load(npy_file)
                # Plot heatmap with 'jet' colormap for high contrast
                im = axes[i+1].imshow(heatmap, cmap='jet')
                axes[i+1].set_title(model)
            else:
                axes[i+1].text(0.5, 0.5, "MISSING", ha='center')
                
            axes[i+1].axis('off')
            
        plt.tight_layout()
        plt.show()

# Run the check for the first 2 images of each dataset
for ds in ALL_DATASETS:
    print(f"\nVALIDATING: {ds}")
    validate_saliency_maps(ds, num_images=2)
def get_word_category(token):
    """
    Categorizes tokens while isolating technical model tokens.
    """
    # Technical infrastructure tokens (Model Sinks)
    technical_tokens = {
        '[CLS]', '[SEP]', '<s>', '</s>', '<|endoftext|>', 
        '<pad>', '[PAD]', '<mask>', '<unk>'
    }
    
    # Standard English Stopwords
    stopwords = {
        'a', 'an', 'the', 'is', 'are', 'was', 'were', 
        'on', 'in', 'at', 'of', 'and', 'with', 'to', 'for'
    }
    
    # Clean BERT (##) and RoBERTa/GPT2 (Ġ) markers
    clean_token = token.replace('##', '').replace('Ġ', '').lower()
    
    if clean_token in technical_tokens:
        return "TECHNICAL"
    if clean_token in stopwords or not clean_token.isalnum():
        return "Grammar/Stopwords"
    return "Content (Nouns/Verbs)"

def calculate_semantic_density_refined(img_idx, dataset_name, text_models):
    saliency_dir = os.path.join(BASE_DIR, f"TextSaliency_{dataset_name}")
    results = []

    for model_label in text_models:
        path = os.path.join(saliency_dir, f"SaliencyText_{model_label}_{img_idx}.npy")
        if os.path.exists(path):
            data = np.load(path, allow_pickle=True).item()
            tokens, scores = data['tokens'], data['scores']
            
            content_sum = 0
            grammar_sum = 0
            
            for t, s in zip(tokens, scores):
                cat = get_word_category(t)
                if cat == "TECHNICAL":
                    continue # Exclude [CLS], [SEP], etc. from the percentage
                elif cat == "Content (Nouns/Verbs)":
                    content_sum += s
                else:
                    grammar_sum += s
            
            total = content_sum + grammar_sum
            if total > 0:
                results.append({
                    "Model": model_label.upper(),
                    "Content (%)": (content_sum / total) * 100,
                    "Grammar (%)": (grammar_sum / total) * 100
                })

    df_density = pd.DataFrame(results)
    
    # Visualization
    ax = df_density.set_index("Model").plot(
        kind='bar', stacked=True, figsize=(10, 6), color=['#2ecc71', '#e74c3c']
    )
    plt.title(f"Refined Semantic Density (Excluding Technical Tokens) | ID: {img_idx}", fontweight='bold')
    plt.ylabel("Percentage of Word-Level Attribution")
    plt.legend(title="Token Type", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
    
    return df_density
def aggregate_to_words(tokens, scores):
    """Aggregates sub-token scores into full word scores for direct comparison."""
    words = []
    word_scores = []
    current_word = ""
    current_score = 0
    count = 0

    for t, s in zip(tokens, scores):
        # Handle BERT (##) and RoBERTa/GPT2 (Ġ) sub-tokens
        is_subtoken = t.startswith("##") or (not t.startswith("Ġ") and count > 0 and "clip" not in t) 
        # Note: Logic varies by tokenizer, this is a generalized heuristic
        
        if is_subtoken:
            current_word += t.replace("##", "")
            current_score = max(current_score, s) # Take the max importance of sub-tokens
        else:
            if current_word:
                words.append(current_word.replace("Ġ", ""))
                word_scores.append(current_score)
            current_word = t
            current_score = s
        count += 1
    
    words.append(current_word.replace("Ġ", ""))
    word_scores.append(current_score)
    return words, word_scores

def calculate_model_correlation(img_idx, dataset_name, text_models):
    """
    Computes Spearman Correlation between the saliency rankings of different models.
    """
    saliency_dir = os.path.join(BASE_DIR, f"TextSaliency_{dataset_name}")
    model_data = {}

    for model in text_models:
        path = os.path.join(saliency_dir, f"SaliencyText_{model}_{img_idx}.npy")
        if os.path.exists(path):
            data = np.load(path, allow_pickle=True).item()
            # Aggregate to words so we compare the same units
            words, scores = aggregate_to_words(data['tokens'], data['scores'])
            model_data[model] = scores

    # Create correlation matrix
    names = list(model_data.keys())
    matrix = np.zeros((len(names), len(names)))

    for i, name1 in enumerate(names):
        for j, name2 in enumerate(names):
            # We compare ranks of words
            s1, s2 = model_data[name1], model_data[name2]
            # Ensure lengths match (truncate to shortest for comparison)
            min_len = min(len(s1), len(s2))
            corr, _ = spearmanr(s1[:min_len], s2[:min_len])
            matrix[i, j] = corr

    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, xticklabels=[n.upper() for n in names], yticklabels=[n.upper() for n in names], cmap="coolwarm", center=0)
    plt.title(f"XAI Rank Correlation (Spearman) | Index: {img_idx}")
    plt.show()
# 1. Define the loading function (Required for the loop)
def load_texts(dataset_name):
    df_path = os.path.join(DATASETS_DIR, f"df_{dataset_name}.pkl")
    if not os.path.exists(df_path):
        raise FileNotFoundError(f"Dataframe not found: {df_path}")
    df_temp = pd.read_pickle(df_path)
    # Get the first caption for each image index
    return [caps[0] if isinstance(caps, list) else str(caps) for caps in df_temp['captions']]

# 2. Your Execution Block (Modified for all 5 models)
text_models_xai = ['bert', 'roberta', 'gpt2', 'clip_text', 'rnn'] # All models
all_datasets = ['Flickr8k', 'Flickr30k', 'ConceptualCaptions']

for ds_name in all_datasets:
    print(f"\n{'='*20}\nProcessing Dataset: {ds_name}\n{'='*20}")
    
    dataset_texts = load_texts(ds_name)
    ds_dir = os.path.join(BASE_DIR, f"TextSaliency_{ds_name}")
    os.makedirs(ds_dir, exist_ok=True)
    
    for model_name in text_models_xai:
        print(f"\n--- Extracting {model_name.upper()} embeddings and saliency ---")
        
        # Use existing _xai functions for Transformers
        if model_name == 'bert':
            feats = get_bert_embeddings_xai(dataset_texts, device, ds_name)
        elif model_name == 'clip_text':
            feats = get_clip_text_embeddings_xai(dataset_texts, device, ds_name)
        elif model_name == 'rnn':
            feats = get_rnn_embeddings_xai(dataset_texts, device, ds_name)
        elif model_name == 'roberta':
            feats = get_roberta_embeddings_xai(dataset_texts, device, ds_name)
        elif model_name == 'gpt2':
            feats = get_gpt2_embeddings_xai(dataset_texts, device, ds_name)
        
        # ADDED: CLIP and RNN support (They need their own logic)
        
        
        # Save features to RAW_DIR for future fusion
        feat_save_path = os.path.join(get_raw_dir(dataset, modality), f"X_text_{model_name}_raw_{ds_name}.npy")
        np.save(feat_save_path, feats)
        
        # Clear VRAM
        del feats
        torch.cuda.empty_cache()
        gc.collect()

    send_ntfy_notification(f"Text XAI completed for {ds_name}.", "TFE Pipeline Update")
calculate_semantic_density_refined(
    img_idx=0,
    dataset_name="Flickr8k",
    text_models=["bert", "roberta", "gpt2", "clip_text"]
)
calculate_model_correlation(
    img_idx=0,
    dataset_name="Flickr8k",
    text_models=["bert", "roberta", "gpt2", "clip_text"]
)